# Delegation and Supervision

**Prerequisites:** `01_single_vs_multi_agent.ipynb`. From `03_tools`: `01_what_is_a_tool.ipynb`

Notebook 1 ran five specialists by hand: each was given its piece, and their pieces were collected. That works when you already know who does what. Most of the time you don't. A message arrives, and somebody has to decide who should deal with it.

A chapter 2 parent cannot make that decision, because it has no list of agents to choose from. A **supervisor** can. It holds a fixed roster of agents, reads each message, asks the one that should handle it, and deals with what comes back.

The recall from notebook 1 is now under way, and messages keep arriving: customers, journalists, the regulator, reports from the field.

**By the end you'll have:**
- Turned an agent into an ordinary chapter 3 tool, and seen that what a supervisor knows about an agent is its tool description, not its persona
- Watched a supervisor route five different messages, and seen what a careless description does to that routing
- Looked at what comes back from a specialist, and what the supervisor does with it
- Seen a supervisor decide who to ask next from what the last agent found
- Compared delegating with handing a conversation over, on the same customer chat

## Setup

`supervisor.py` now holds `Agent` (moved from notebook 1), plus what this notebook adds. `recall.py` holds the recall: the brief, the specialists' personas, and a small database behind their tools. `meter.py` is notebook 1's `Meter`.

## The roster

Four specialists from notebook 1, each with the brief, and now with real tools where their job needs one. Support can look up a serial number. The engineer can find which battery cell lot a unit was built with, and what testing found for that lot. The database is small and made up, but what it returns is fixed, so every run is about the same facts.

Each agent now has two pieces of text. Its **persona** is its system prompt: how to *be* this agent. Its **description** is what other agents are told about it: when to *ask* it. They are written for different readers.

That last pair is the catch the database is built around. Unit AR2‑2409‑01544 is from September's batch, which is *not* in the recall, but it was built with cells from the same lot as the recalled batch. Support's lookup cannot see that. The engineer's can. Section 4 depends on it.

## An agent is a tool

A supervisor needs a way to ask another agent something and get an answer back. Chapter 3 already has exactly that shape: a `Tool` is a function with a description, and the model decides when to call it. So an agent becomes a tool with one argument:

```python
def as_tool(agent):
    def ask(request: Annotated[str, "Everything this specialist needs to know to do the job. "
                                    "They see nothing else: not the original message, not other replies."]) -> str:
        return agent.run(request)
    return Tool(ask, name=f"ask_{agent.name}", description=agent.description)
```

Here is everything a supervisor's model is told about the engineer:

Three things follow from this.

- **The description is the only thing routing can go on.** The engineer's persona, its bluntness, its tools, the brief: none of it is in the schema. A supervisor decides who to ask from two sentences somebody wrote about the agent.
- **The request is the whole of what the agent will know.** It is chapter 2's self-contained goal again, and the parameter's description says so, because the supervisor writes that request on the spot.
- **Nothing new was needed.** The supervisor below is an ordinary `Agent` whose tools happen to be other agents. It runs chapter 2's loop unchanged: `Think` picks a tool, `Act` calls it, which runs a whole other agent, and `Observe` puts the reply back.

## Choosing who to ask

Five messages of the kind a recall desk gets. For each, only the supervisor's *first* decision is taken: which specialist it asks, and what it asks them. Nothing is run beyond that, so this costs one call per message.

Three of the five go where you would expect: the customer to support, the regulator to the lawyer, the lab measurement to engineering.

**The journalist went to engineering.** PR's description says "journalists and the public", and it was not chosen. The supervisor routed by *what the question needs looking up* rather than by *who is asking*. The social post went to engineering too, in this run. In an earlier run of the same cell it went to PR, which is the first hint that one run per message does not settle anything. Later in this notebook the same question gets measured properly.

Look at the requests as well as the choices. Each one restates the message in the supervisor's own words, because the specialist will see nothing else. When that restatement drops something, the specialist works from less than the supervisor had. There is an example of that below.

### A careless description

Descriptions get written once and then forgotten. Here is a plausible one for PR: *"Communications. Writes anything that goes outside the company about the recall."* It is not wrong. PR does write for outside the company. But so do support, who write to customers, and the lawyer, who writes to the regulator.

The routing came out the same as before: support, engineering, the lawyer, engineering, engineering. On this evidence the careless description changed nothing.

That is a weak statement, and it should be. One run per message cannot show a small change in routing, in either direction. To say anything, the same message has to be run many times and the choices counted, which is what the next-but-one section does for a different change. It costs a call per trial, so it is worth doing for the routing you depend on, not for every description.

## What comes back

Now one message all the way through: the journalist's. The supervisor asks a specialist, the specialist does its work, and a reply comes back to the supervisor, which then gives its own final answer. The log shows what the specialist actually wrote, which is otherwise buried in the supervisor's conversation.

Two things are worth pulling out of that.

**PR was never asked.** The supervisor asked engineering, then support, and wrote the reply to the journalist itself. Its own prompt says it never writes replies, and what it produced is close to a paraphrase of support's answer. So the reply that would go to a journalist was written in the supervisor's voice by way of a customer-support answer, which is exactly the mixing notebook 1 spent its length on.

**A reply is a plain string.** Support wrote *"no other batches are affected"*, and the supervisor passed it on. The database disagrees: the first 1,800 units of batch AR2‑2409 were built from the same cell lot as the recalled batch. Support cannot see that; only engineering's lookup can. Nothing in what came back could say *"this is the part I am sure about"*, *"this came from the brief, not from a tool"*, or *"do not send this outside the company"*. A string has nowhere to put any of it, and the supervisor has nothing to check. That gap is what notebook 3 is about.

## Deciding who's next from what came back

Two customer emails that look almost the same. Both say a unit got hot while charging. Neither unit is from the recalled batch. The supervisor gets each one and decides everything after that itself.

Both reports went to support and stopped there.

For report A this is the dangerous one. The customer says the case has **bulged**, which is the same symptom as the recall. Support looked the serial up, saw it was not on the recall list, and answered:

> "So your headphones aren't part of the recall. You can keep using them on battery power… **For now, you're safe.**"

The unit is from batch AR2‑2409, which is not recalled, but `cell_lot` says it was built from lot VC‑7731 — the lot behind the recall. Support has no way to see that; `lookup_serial` only answers "is this serial on the list". Engineering has the lookup that would have caught it, and engineering was never asked.

Report B went the same way, and support's answer contradicts itself: it opens with "Stop charging your headphones right now" and then says warmth is probably normal.

The supervisor routed by **who sent the message**. A customer email goes to customer support. Nothing in what it was given made the content of the message — bulging, overheating — count for more than the sender.

### Saying what an agent cannot do

A description says when to ask an agent. None of these say when *not* to, and that is the gap report A fell into. Support's description promises "whether their unit is affected", which sounds like the whole question. It is really "whether this serial is on the recall list", which is a much smaller thing, and the difference is invisible from the outside.

So each description gets one more sentence, saying where the agent's knowledge stops and who to ask instead. Nothing else changes: same agents, same personas, same tools, same supervisor prompt.

With the extra sentences, this run did reach engineering, though not first. Support was still asked first and still answered *"not part of the recall… You can keep using the headphones normally"*, and only then did the supervisor ask engineering, which found lot VC‑7731 and the separator defect. The final answer to the customer is the right one: stop charging, return it, replacement or refund.

Report B shows two other things, both worth having.

- **The request is the whole of what a specialist sees.** The supervisor asked engineering *"Assess whether overheating during charging with a 65 W charger is a known defect for this serial"* — without the serial. Engineering answered *"I need the specific serial number to look up its cell lot."* A whole round trip was spent on a missing word.
- **A supervisor's budget is spent in whole agents.** After that wasted round trip the run hit `max_iterations=8` and ended with *"Stopping: reached the maximum of 8 iterations… This is partial progress, not a final answer."* The customer would have got nothing. Chapter 2's budgets count iterations of the supervisor's own loop, and each of those iterations can be an entire other agent's run. Two things about that limit are worth knowing: `Runtime` counts every *component* it runs, and this loop has three, so 8 was under three turns — much tighter than it looks. `Agent`'s default was raised to 24 while notebook 3 was being written; this run is left as it happened. So is its ending. Chapter 2's `Runtime` has since changed what a stop does: the supervisor now gets one last turn, with no tools, to reply from what its specialists had already found, and the stop is recorded on `conversation.stopped` rather than in the text. The customer would get a reply — one that reads like a finished answer, which is exactly why notebook 5 checks `stopped` and not the words.

So: better than before on report A, and nothing conclusive yet. Whether the descriptions caused the improvement or the run was lucky is not something one run can answer.

### The same rule, one level up

The descriptions did not move it. The other place a routing rule can live is the supervisor's own instructions, which it reads on every message, rather than in a tool schema it reads while choosing. Same rule, same agents, same sharper descriptions; only the supervisor's prompt changes.

With the rule in the supervisor's prompt, report A went to engineering **first**: cell lot VC‑7731, the separator defect, not in the recalled batch but the same fault is possible. The supervisor then asked support to write the customer's reply, which opens *"Stop charging the headphones right now"* and explains why. That is the answer the first version should have given.

Report B went to engineering first as well, took four calls, and came back with the right answer the other way: lot VC‑7702, tested clean, *"It's not a sign of the recall defect"*, with advice to watch it. The rule costs an extra agent call on messages that do not need one, which is what buying a guarantee looks like.

### How often, not whether

Every trace above is one run, and one run of a model is a coin flip rather than a result. The question worth answering is how *often* each version asks engineering first about report A, so each is run eight times. Only the supervisor's first decision is taken, which is the decision in question and costs one call.

| | engineer asked first |
|---|---|
| plain descriptions | 1 of 8 |
| sharper descriptions | 0 of 8 |
| the rule in the supervisor's prompt | **8 of 8** |

Sharper descriptions against plain: p = 1.0, which is no effect at all. The rule against either: p ≤ 0.001.

**The two sentences added to the descriptions did nothing to the first choice**, and the earlier trace that looked like an improvement was a coin landing the other way. This is why the traces above are labelled as examples: run each of them again and they can come out differently.

Be precise about what was measured. This counts the **first** specialist asked. The sharper run did eventually reach engineering, on its second step, so a description can still change what happens later in a run. What it did not do is change who gets asked first, which is the decision that let report A be answered by support alone.

Why the difference. The supervisor's prompt is read on every message, as part of who the supervisor is. A tool description is read while choosing between tools, and on a model this size it carries much less weight. **A rule that must always hold belongs in the supervisor's instructions; a description says what an agent is for, and cannot be relied on to carry a rule.**

**This is measured on one model.** Everything here runs on gpt-oss-20b, and how much weight a model gives a tool description against its own system prompt is very likely a matter of size and training. A larger model may read the descriptions properly and need no rule at all. What carries across is the method rather than the number: if routing matters, put the rule where your model actually reads it, and count the choices rather than assuming.

The 24 trials cost 22,670 tokens, about the same as two full traced runs. Measuring routing is not free, but it costs less than trusting it.

## Handing over instead of asking

Everything so far is **delegation**: the supervisor asks, gets an answer, and stays in charge. Every message comes to it first, and every reply goes back through it.

A **handoff** moves the conversation instead. The agent that hands over steps out, and the one it hands to talks to the person directly from then on, seeing everything said so far. `Chat` in `supervisor.py` does this with one more tool per target, `transfer_to_<name>`. Calling it stops the current agent's loop through the `cancel` hook chapter 2 uses to stop abandoned branches, and the new owner answers the same turn.

The same three-message customer chat, both ways. First a front desk that hands over:

And a front desk that delegates. It keeps the conversation, asks support whenever it needs to, and passes the answer on:

What support was given to work from, each way. With the handoff, support saw the conversation itself. With delegation, it saw only what the front desk wrote in each request:

**The handoff.** The front desk transferred on the very first message, with a note for support. From then on support talked to the customer directly: it asked for the serial, looked it up, and answered in its own voice. Five calls, 4,891 tokens.

**The delegation.** The front desk stayed in charge for all three turns, asked support each time, and rewrote every answer. Eleven calls, 12,178 tokens: **2.5×** the handoff for the same conversation.

It also drifted. Its first reply, before anyone had a serial number, told the customer:

> "**If you're not in the recall batch** — keep using the headphones as you normally would. They're safe on battery; just don't charge them."

Nobody said that. "Don't charge them" is advice for a recalled unit, applied to units that are not recalled. The more a front desk rewrites, the more chances it has to invent, and it rewrites on every turn.

**What each side saw.** After the handoff, support had the whole transcript, the customer's words included. With delegation, support saw three separate requests, each one the front desk's summary. The third one did carry the serial number forward, because the front desk chose to write it in; nothing made it.

**What each is good for.** A handoff suits a conversation that belongs to one specialist from the moment you know who it is: cheaper, fewer hands, the specialist's own voice. Delegation suits work where the requests keep changing who they need, because the supervisor is still there to send the next one somewhere else. After a handoff, the front desk is out of the conversation, and the new owner needs its own handoffs to pass it on again.

## What it cost

- **Routing itself is cheap.** One decision is about 600 to 1,200 tokens. Everything expensive is the work that follows it, so the thing worth getting right is which agent the work goes to.
- **Each extra specialist is a whole agent run.** Report B under the rule is four calls; report A under it is eight. Notebook 1's fork paid for branches that were all the same agent; here you pay for the one you ask, plus a supervisor call each time round.
- **Delegation costs more than a handoff per turn of a conversation**, 12,178 against 4,891 here, because the supervisor pays to read the conversation and to rewrite the answer every turn.
- **The measurement cost 22,670 tokens**, more than any single trace in this notebook. It is also the only number here that supports a claim.

## Where this leaves us

A supervisor is an ordinary agent whose tools are other agents. That is all the machinery needed, and chapter 2's loop runs it unchanged. What the notebook found beyond that:

- **A supervisor knows an agent by its description**, not by its persona, its tools or what it can see. Write the description for the reader that has to choose.
- **The request is the whole of what the chosen agent gets.** Leave the serial number out and the specialist asks for it, at the price of another agent run.
- **Routing you depend on belongs in the supervisor's instructions.** Measured over eight trials each: a sharpened description changed the first choice 0 of 8 times, a rule in the prompt 8 of 8.
- **Delegation keeps the supervisor in charge; a handoff gives the conversation away.** The handoff was 2.5× cheaper here and kept the specialist's own voice; delegation kept the ability to send the next turn elsewhere.
- **Results come back as plain strings.** Nothing in a reply can say "I am not sure", "this is internal", or "ask engineering about this part", so the supervisor cannot check any of it. Notebook 3 gives replies a shape.

Two failures are worth carrying forward to notebook 5, where failure handling is the subject: a run that ended on *"reached the maximum of 8 iterations"* with nothing for the customer, and a supervisor that answered a journalist with a claim the database contradicts, because it never asked the agent that could have checked.